# Milestone 2: diagnose the frozen baseline

The first baseline passed **two of six validation gates**. This notebook evaluates that unchanged epoch-10 checkpoint on **training and validation only**, to investigate its errors. It performs no training, recurrence sweep, or test-split evaluation.

Attach the original `milestone2_baseline_artifacts.zip` (or its extracted directory). Enable GPU and internet access. Push these notebook/package changes first, review settings, then run all cells in order. The source checkpoint must match the recorded hash; results cannot silently come from a different model.


## 1. Settings


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
# Use the new diagnostic code revision, not the older training revision.
REPO_REF = "milestone2"
REPO_DIR = "/kaggle/working/multi-modal-loop-diagnostics"
RUN_ROOT = "/kaggle/working/milestone2_diagnostics"
BASELINE_SOURCE = "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_baseline_artifacts.zip"

## 2. Checkout and setup

Uses one GPU (`cuda:0`) and preserves the installed PyTorch stack. An existing checkout must match the requested revision. Use a new checkout path or restart the kernel when changing code revisions.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## 3. Audit source and prepare fresh output

Validates the checkpoint hash, embedded manifest, settings, budget, original control report, and acceptance record before inference. ZIP members are validated before extraction. Both the training revision and diagnostic revision are recorded. Original artifacts are never overwritten.


In [ ]:
import multimodal_loop.eval.kaggle_diagnostics as diagnostic_helpers
from multimodal_loop.eval.kaggle_diagnostics import (
    diagnostic_archive,
    prepare_diagnostics,
    run_diagnostics,
)

if not Path(diagnostic_helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")

run = prepare_diagnostics(REPO_DIR, RUN_ROOT, BASELINE_SOURCE)

## 4. Frozen-model diagnosis

Runs the existing question-only forward path at R=2 and batch size 32 on all 9,216 training and 2,304 validation QA examples. Metadata is attached only for reporting after predictions are collected. Logs stream below and are retained in the run directory.


In [ ]:
report = run_diagnostics(run)

## 5. Inspect the breakdowns

Tables group errors by geometry, anchor shape, direction, and target position. The position confusion table maps predicted colors to objects, the absent fourth color, or invalid tokens. Training metrics here evaluate the final frozen checkpoint; historical training losses averaged predictions during optimization and are not the same measurement.

A large training/validation gap supports a generalization problem. Group differences help localize errors but do not by themselves establish why they occur. The four validation geometries are correlated groups, not thousands of independent layouts.


In [ ]:
import html

from IPython.display import HTML, display


def show_table(title, rows):
    columns = list(dict.fromkeys(key for row in rows for key in row))
    parts = [f"<h3>{html.escape(title)}</h3><table><tr>"]
    parts += [f"<th>{html.escape(key)}</th>" for key in columns]
    parts.append("</tr>")
    for row in rows:
        parts.append("<tr>")
        for key in columns:
            value = row.get(key, "")
            text = f"{value:.6f}" if isinstance(value, float) else str(value)
            parts.append(f"<td>{html.escape(text)}</td>")
        parts.append("</tr>")
    display(HTML("".join(parts) + "</table>"))


show_table(
    "Frozen checkpoint: train versus validation",
    [
        {
            "split": split,
            **{
                key: result[key]
                for key in ("total", "correct", "accuracy", "loss", "invalid_predictions")
            },
            "all_four_accuracy": result["all_four"]["accuracy"],
            "different_answer_pair_accuracy": result["different_answer_pairs"]["accuracy"],
        }
        for split, result in report["splits"].items()
    ],
)
for split, result in report["splits"].items():
    for axis, rows in result["breakdowns"].items():
        if axis == "target_position_confusion":
            rows = [{"target_position": key, **counts} for key, counts in rows.items()]
        show_table(f"{split}: {axis}", rows)
show_table(
    "Validation comparison with original report",
    [
        {"measurement": name, **report["validation_comparison"][name]}
        for name in ("original", "diagnostic", "differences")
    ],
)
print("Exact original-validation match:", report["validation_comparison"]["exact_match"])

## 6. Inspect selected mistakes

The HTML shows eight highest-confidence incorrect QA examples per split (ties use stored example index), with each image's four question predictions. The same image may appear more than once. These deliberately selected mistakes illustrate failure cases; use the full tables to assess their frequency.


In [ ]:
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))

## 7. Retain diagnostic artifacts

Download the new archive using the link or notebook output files, and bring it back for review. It contains summary JSON, every train/validation prediction in JSONL, the HTML inspection page, source/code/runtime provenance, and logs. Staged copies of the original baseline are excluded; retain the original archive separately. A rerun requires a fresh output directory. No cell continues training or evaluates the reserved test split.


In [ ]:
from IPython.display import FileLink

archive = diagnostic_archive(run)
print("Diagnostic archive:", archive)
display(FileLink(str(archive)))